In [6]:
import os
from datetime import datetime, timezone
from typing import Any, Dict, List

import pandas as pd
import streamlit as st
from langsmith import Client


LANGSMITH_KEY = os.getenv("LANGCHAIN_API_KEY") or os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGCHAIN_PROJECT", "default")


def get_langsmith_client() -> Client:
    if not LANGSMITH_KEY:
        raise RuntimeError("Missing LANGCHAIN_API_KEY or LANGSMITH_API_KEY in environment.")
    return Client(api_key=LANGSMITH_KEY)


def _safe_iso(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, datetime):
        if value.tzinfo is None:
            value = value.replace(tzinfo=timezone.utc)
        return value.isoformat()
    return str(value)


def fetch_langsmith_traces(project_name: str, limit: int = 100) -> pd.DataFrame:
    client = get_langsmith_client()
    runs = list(client.list_runs(project_name=project_name, limit=max(1, int(limit))))

    rows: List[Dict[str, Any]] = []
    for run in runs:
        rows.append(
            {
                "run_id": str(getattr(run, "id", "")),
                "name": str(getattr(run, "name", "")),
                "run_type": str(getattr(run, "run_type", "")),
                "status": str(getattr(run, "status", "")),
                "error": str(getattr(run, "error", "")) if getattr(run, "error", None) else "",
                "start_time": _safe_iso(getattr(run, "start_time", None)),
                "end_time": _safe_iso(getattr(run, "end_time", None)),
                "latency_ms": float(getattr(run, "latency", 0.0) or 0.0),
                "total_tokens": int(getattr(run, "total_tokens", 0) or 0),
                "project_name": project_name,
            }
        )

    if not rows:
        return pd.DataFrame(
            columns=[
                "run_id",
                "name",
                "run_type",
                "status",
                "error",
                "start_time",
                "end_time",
                "latency_ms",
                "total_tokens",
                "project_name",
            ]
        )

    frame = pd.DataFrame(rows)
    return frame.sort_values(by="start_time", ascending=False, na_position="last").reset_index(drop=True)


print("LangSmith trace utilities loaded. Use fetch_langsmith_traces(project_name, limit).")

LangSmith trace utilities loaded. Use fetch_langsmith_traces(project_name, limit).


In [7]:
def render_langsmith_traces_tab(default_project: str = LANGSMITH_PROJECT) -> None:
    st.subheader("LangSmith Traces")
    st.write("View latest traces in Streamlit and inspect run details.")

    controls = st.columns(4)
    project_name = controls[0].text_input("Project", value=default_project, key="ls_project")
    limit = controls[1].slider("Rows", min_value=10, max_value=500, value=100, step=10, key="ls_limit")
    auto_refresh = controls[2].checkbox("Auto refresh", value=False, key="ls_auto_refresh")
    refresh_clicked = controls[3].button("Refresh", key="ls_refresh", use_container_width=True)

    if auto_refresh or refresh_clicked or "ls_traces_df" not in st.session_state:
        try:
            st.session_state["ls_traces_df"] = fetch_langsmith_traces(project_name=project_name, limit=limit)
            st.session_state["ls_error"] = ""
        except Exception as exc:
            st.session_state["ls_error"] = str(exc)
            st.session_state["ls_traces_df"] = pd.DataFrame()

    err = st.session_state.get("ls_error", "")
    if err:
        st.error(f"Failed to load traces: {err}")
        return

    traces_df = st.session_state.get("ls_traces_df", pd.DataFrame())
    if traces_df.empty:
        st.info("No traces found for this project yet.")
        return

    st.dataframe(traces_df, use_container_width=True, hide_index=True)

    st.markdown("**Trace Detail**")
    run_ids = traces_df["run_id"].astype(str).tolist()
    selected_run = st.selectbox("Select run", options=run_ids, key="ls_selected_run")
    selected_row = traces_df[traces_df["run_id"].astype(str) == str(selected_run)].head(1)
    if not selected_row.empty:
        st.json(selected_row.to_dict(orient="records")[0])

In [8]:
# Example usage inside your Streamlit app:
#
# tab_planner, tab_langsmith = st.tabs(["Planner", "LangSmith Traces"])
# with tab_langsmith:
#     render_langsmith_traces_tab()
#
# For quick local test in an existing Streamlit script, call:
# render_langsmith_traces_tab()

print("Ready. Import these functions into your Streamlit app and call render_langsmith_traces_tab() in a dedicated tab.")

Ready. Import these functions into your Streamlit app and call render_langsmith_traces_tab() in a dedicated tab.
